# Agentic Orchestrator

Welcome to the Agentic AI 
Instead of just answering questions, the AI now acts as an **Autonomous Agent** with "Tools".

**How it works:**
1. You ask a complex question.
2. The Agent *thinks* about how to solve it.
3. It decides which tools to use:
   - 📚 **RAG Search** (Your ChromaDB of legal PDFs)
   - 🌐 **Web Search** (For live news/updates not in your PDFs)
   - 📊 **Risk Scorer** (Calculates Biopiracy risk automatically)
4. It uses the tools, reads the results, and writes a final, comprehensive answer.

In [1]:
!pip install -q groq chromadb sentence-transformers duckduckgo-search rich

In [2]:
!pip install -q ddgs

In [3]:
import os
from pathlib import Path

os.environ["GROQ_API_KEY"] = ""

PROJECT_ROOT   = Path("/Users/ritesh/Downloads/IP_Shakti_Sahayak")
CHROMA_DB_PATH = PROJECT_ROOT / "chromadb_store"

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
COLLECTION_NAME = "ip_sakti_legal_corpus"
LLM_MODEL = "openai/gpt-oss-120b" 

print("✅ Configuration loaded.")

✅ Configuration loaded.


## Load Databases & Clients

In [4]:
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import json

# Load Groq
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Load ChromaDB
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))
collection = chroma_client.get_collection(name=COLLECTION_NAME)

# Load Embedding Model
print("⏳ Loading embedding model...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL, device="mps")

print("✅ All clients loaded and ready.")

⏳ Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ All clients loaded and ready.


## Define Tool 1: RAG Search (The Legal Brain)
packaged into a "Tool" function.

In [5]:
def search_legal_database(query: str, limit: int = 5) -> str:
    """Searches the internal offline ChromaDB for Indian and International IP Acts, Rules, and Guidelines."""
    print(f"   [Tool Executing] 📚 Searching legal database for: '{query}'")
    
    query_embedding = embedding_model.encode(query, normalize_embeddings=True).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=limit,
        include=["documents", "metadatas"]
    )
    
    if not results["documents"] or not results["documents"][0]:
        return "No relevant legal documents found in the internal database."
    
    formatted_results = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        source = meta.get("source_file", "Unknown")
        formatted_results.append(f"[Source: {source}]\n{doc}\n")
        
    return "\n---\n".join(formatted_results)

## Define Tool 2: Web Search (For breaking news)
Allows the agent to search the live internet if it needs recent information not in the PDFs.

In [6]:
from ddgs import DDGS
import warnings

# Suppress annoying warnings
warnings.filterwarnings('ignore')

def search_internet(query: str) -> str:
    """Searches the live internet for recent news, updates, or general information."""
    print(f"   [Tool Executing] 🌐 Searching the internet for: '{query}'")
    try:
        results = DDGS().text(query, max_results=3)
        
        if not results:
            return "No internet results found."
            
        formatted = []
        for r in results:
            # DDGS returns slightly different dictionary keys now
            title = r.get('title', 'No Title')
            body = r.get('body', r.get('snippet', 'No Snippet'))
            link = r.get('href', r.get('link', 'No Link'))
            formatted.append(f"Title: {title}\nSnippet: {body}\nLink: {link}")
            
        return "\n\n".join(formatted)
    except Exception as e:
        return f"Internet search failed: {str(e)}"

## Define Tool 3: Biopiracy Risk Scorer 
A deterministic tool that calculates risk based on a hardcoded algorithm.

In [7]:
def calculate_biopiracy_risk(ingredients: list, is_classical_text_referenced: bool, has_modern_processing: bool) -> str:
    """Calculates the biopiracy and patent rejection risk for an Ayurvedic formulation."""
    print(f"   [Tool Executing] 📊 Calculating Biopiracy Risk for ingredients: {ingredients}")
    
    risk_score = 0
    reasons = []
    
    # Rule 1: Using known classical herbs
    known_herbs = ['ashwagandha', 'brahmi', 'neem', 'turmeric', 'tulsi', 'amla']
    matched_herbs = [herb for herb in ingredients if herb.lower() in known_herbs]
    if matched_herbs:
        risk_score += 40
        reasons.append(f"Contains highly documented traditional herbs ({', '.join(matched_herbs)}). High risk of Section 3(p) rejection.")
        
    # Rule 2: Classical text reference
    if is_classical_text_referenced:
        risk_score += 30
        reasons.append("Explicitly references classical Ayurvedic texts. Likely already in the TKDL.")
        
    # Rule 3: Modern processing (Mitigates risk)
    if has_modern_processing:
        risk_score -= 20
        reasons.append("Uses modern processing (e.g., nanoparticles, novel extraction). This adds an inventive step, lowering patent rejection risk.")
    else:
        risk_score += 20
        reasons.append("No modern technical intervention claimed. Just a mere admixture of known ingredients (Section 3(e) risk).")
        
    # Clamp score between 0 and 100
    risk_score = max(0, min(100, risk_score))
    
    band = "🟢 LOW RISK (File Patent)" if risk_score <= 30 else "🟡 MEDIUM RISK (Narrow Claims)" if risk_score <= 60 else "🔴 HIGH RISK (Defensive Publish)"
    
    return json.dumps({
        "risk_score": risk_score,
        "risk_band": band,
        "analysis": reasons
    }, indent=2)

## The Agent Orchestrator (The Brain)
This function tells the LLM what tools it has, and runs a loop so the LLM can decide which tool to call.

In [8]:
# 1. Define the Tools Schema for the LLM
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "search_legal_database",
            "description": "Searches the internal offline database for Indian and International IP Acts, Rules, and Guidelines.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query (e.g., 'patentability of turmeric')"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            # Renamed to prevent the LLM from hallucinating arguments
            "name": "perform_web_search", 
            "description": "Searches the live internet. Use this for recent news or current officials.",
            "parameters": {
                "type": "object",
                "properties": {
                    "search_query": {"type": "string", "description": "The exact phrase to search on the web"}
                },
                "required": ["search_query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_biopiracy_risk",
            "description": "Calculates the biopiracy and patent rejection risk for a specific Ayurvedic formulation.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ingredients": {"type": "array", "items": {"type": "string"}, "description": "List of herbs/ingredients"},
                    "is_classical_text_referenced": {"type": "boolean", "description": "True if based on ancient texts"},
                    "has_modern_processing": {"type": "boolean", "description": "True if using nanoparticles, etc."}
                },
                "required": ["ingredients", "is_classical_text_referenced", "has_modern_processing"]
            }
        }
    }
]

# 2. The Agent Loop
def run_agent(user_prompt: str):
    print(f"\n{'='*70}")
    print(f"🧑 USER: {user_prompt}")
    print(f"{'='*70}")
    
    messages = [
        {
            "role": "system", 
            "content": "You are the IP-SAKTI Agent. You have tools to search legal databases, search the web, and calculate biopiracy risk. Decide which tools to use. You can use multiple tools sequentially. Always synthesize a final answer."
        },
        {"role": "user", "content": user_prompt}
    ]
    
    for turn in range(5):
        response = groq_client.chat.completions.create(
            model=LLM_MODEL,
            messages=messages,
            tools=tools_schema,
            tool_choice="auto",
            max_tokens=4096
        )
        
        response_message = response.choices[0].message
        tool_calls = response_message.tool_calls
        
        if not tool_calls:
            print(f"\n🤖 IP-SAKTI FINAL ANSWER:\n{response_message.content}")
            break
            
        # Append the assistant's response so it remembers it called a tool
        messages.append(response_message) 
        
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            
            # Safely parse arguments
            try:
                args = json.loads(tool_call.function.arguments)
            except:
                args = {}
            
            if function_name == "search_legal_database":
                tool_result = search_legal_database(args.get("query", ""))
            elif function_name == "perform_web_search":
                # Using the renamed argument
                tool_result = search_internet(args.get("search_query", ""))
            elif function_name == "calculate_biopiracy_risk":
                tool_result = calculate_biopiracy_risk(
                    args.get("ingredients", []), 
                    args.get("is_classical_text_referenced", False), 
                    args.get("has_modern_processing", False)
                )
            else:
                tool_result = "Error: Tool not found."
                
            messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": tool_result,
            })

## Test 1: The Agent figures out to use the Risk Scorer

In [9]:
run_agent("I want to patent a traditional mixture of Neem and Tulsi exactly as described in the Charaka Samhita. Calculate my biopiracy risk and tell me what the Indian Patent Act says about this.")


🧑 USER: I want to patent a traditional mixture of Neem and Tulsi exactly as described in the Charaka Samhita. Calculate my biopiracy risk and tell me what the Indian Patent Act says about this.
   [Tool Executing] 📊 Calculating Biopiracy Risk for ingredients: ['Neem', 'Tulsi']
   [Tool Executing] 📚 Searching legal database for: 'Indian Patent Act Section 3(k) traditional knowledge neem tulsi patentability Charaka Samhita'

🤖 IP-SAKTI FINAL ANSWER:
**1. Biopiracy‑risk assessment (calculated)**  

| Parameter | Value |
|-----------|-------|
| Ingredients | Neem (*Azadirachta indica*), Tulsi (*Ocimum tenuiflorum*) |
| Reference to classical text (Charaka Samhita) | **Yes** |
| Modern processing / technical transformation | **No** |
| **Risk score** | **90 / 100** |
| **Risk band** | 🔴 **HIGH RISK (Defensive Publish)** |

*Why the score is high*  

1. **Both herbs are fully documented in the Traditional Knowledge Digital Library (TKDL).** The antiseptic, anti‑inflammatory and pesticidal p

## Test 2: The Agent figures out to use the Web Search

In [10]:
run_agent("Who is the current head of the Ministry of AYUSH in India right now? Check the internet.")


🧑 USER: Who is the current head of the Ministry of AYUSH in India right now? Check the internet.
   [Tool Executing] 🌐 Searching the internet for: 'current Minister of AYUSH India'
   [Tool Executing] 🌐 Searching the internet for: 'Prataprao Jadhav Minister of AYUSH 2024'

🤖 IP-SAKTI FINAL ANSWER:
**Current head of the Ministry of AYUSH (India)**  
- **Name:** **Prataprao Ganpatrao Jadhav**  
- **Position:** **Minister of AYUSH, Government of India** (appointed in July 2024)  

**Sources**  

1. Wikipedia entry for the Ministry of AYUSH lists Prataprao Jadhav as the incumbent minister【perform_web_search†1】.  
2. A recent Wikipedia biography confirms that “Prataprao Ganpatrao Jadhav … is serving as the 3rd Minister of AYUSH since 2024”【perform_web_search†2】.  

Thus, as of the latest publicly available information (July 2024), the Ministry of AYUSH is headed by **Minister Prataprao Ganpatrao Jadhav**.


## Test 3: Multi-Tool Intelligence (The Ultimate Test)

In [11]:
run_agent("I have a new drug made of Ashwagandha using Nanoparticle extraction. First, calculate my risk score. Then search the legal database for rules on 'nanoparticles'. Synthesize a final IP strategy for me.")


🧑 USER: I have a new drug made of Ashwagandha using Nanoparticle extraction. First, calculate my risk score. Then search the legal database for rules on 'nanoparticles'. Synthesize a final IP strategy for me.
   [Tool Executing] 📊 Calculating Biopiracy Risk for ingredients: ['Ashwagandha']
   [Tool Executing] 📚 Searching legal database for: 'nanoparticles'
   [Tool Executing] 📚 Searching legal database for: 'nanoparticles patent India'
   [Tool Executing] 🌐 Searching the internet for: 'India patent guidelines nanoparticles'

🤖 IP-SAKTI FINAL ANSWER:
**1. Biopiracy / Patent‑Rejection Risk Score**

| Parameter | Value |
|-----------|-------|
| **Modern processing (nanoparticle extraction)** | ✅ Yes |
| **Ingredient(s)** | Ashwagandha (Withania somnifera) |
| **Reference to classical Ayurvedic texts** | ❌ No (not indicated) |
| **Calculated risk score** | **20 / 100** |
| **Risk band** | 🟢 **LOW RISK – file a patent** |

*Why the score is low*  
- The herb itself is well‑known in Indian 

In [13]:
run_agent("I developed a completely new Ayurvedic formulation using five medicinal plants. None of the exact formulation is mentioned in the classical texts. What IP protection should I consider?")


🧑 USER: I developed a completely new Ayurvedic formulation using five medicinal plants. None of the exact formulation is mentioned in the classical texts. What IP protection should I consider?
   [Tool Executing] 📚 Searching legal database for: 'patentability Ayurvedic formulation Indian Patent Act guidelines'
   [Tool Executing] 📊 Calculating Biopiracy Risk for ingredients: ['HerbA', 'HerbB', 'HerbC', 'HerbD', 'HerbE']

🤖 IP-SAKTI FINAL ANSWER:
**Short answer:**  
Treat the formulation as a *product‑invention* and, if you have a novel process or a new therapeutic use, also as a *process‑ or use‑invention*.  File a **patent** in India (and, if you need overseas protection, a PCT application that later enters the national phase of the countries of interest).  In parallel register a **trademark** for the brand name, consider a **geographical indication (GI)** if the product is tied to a specific region, and keep any unrevealed “secret sauce” as a **trade‑secret** until you are ready to 